# Causal Inference in Practice
## Week 15 — Capstone, Reproducibility & Communication · Practice Notebook

> **Block IV — Modern methods & application**
>
> Put it together: a defensible, reproducible causal analysis you can explain to a decision-maker.

**How to use this notebook.** Run the cells top to bottom. Sections marked
**🔧 Exercise** contain a `# TODO` for you to complete; a matching
**✅ Solution** cell follows (collapsed in spirit — try it yourself first).
Every dataset here is *simulated with a known ground truth*, so you can always
check whether your estimate recovered the right answer.

*Estimated time: 60–90 minutes. Toolkit: `numpy`, `pandas`, `statsmodels`,
`scikit-learn`, `matplotlib` — all standard.*

---


In [ ]:
# --- Environment check & shared setup -------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.figsize": (7, 4.2), "axes.grid": True,
                     "grid.alpha": 0.25, "font.size": 11})

RNG = np.random.default_rng(7)   # one seed for the whole notebook → reproducible
print("Environment OK — numpy", np.__version__, "| pandas", pd.__version__)

# Mini capstone: one decision problem, the whole workflow

This notebook is a **template for your capstone**. We invent a single decision problem with a **known true effect**, then walk every step of the course workflow on it:

1. **State the estimand**
2. **Encode assumptions** as a DAG-shaped data-generating process
3. **Identify** the back-door adjustment set
4. **Estimate** with **two** methods and check they agree *and* recover the truth
5. **Validate** with a robustness / multiverse loop and a sensitivity (E-value) check

…and finish with a plain-language **communicate-the-result** summary and a **reproducibility** log. Swap in your real data and this becomes your submission.

## 0 · Reproducibility — fixed seed & logged environment

Reproducibility starts here: one seed (`RNG`, defined in the setup cell — we **reuse** it, never reseed) and a printed log of library versions so the result travels with the environment that produced it.

In [ ]:
import sys
import numpy as np, pandas as pd
import statsmodels.api as sm
import sklearn, statsmodels

print('--- environment log (paste this into your report) ---')
print('python     ', sys.version.split()[0])
print('numpy      ', np.__version__)
print('pandas     ', pd.__version__)
print('scikit-learn', sklearn.__version__)
print('statsmodels ', statsmodels.__version__)
print('seed        7  (RNG = np.random.default_rng(7), reused)')

## 1 · State the estimand

**Decision problem.** A company offers an optional *training program* `D` to employees and wants to know its effect on *productivity* `Y`. Enrollment is **not** random: more experienced and more motivated people enroll, and those traits also raise productivity. That is confounding.

**Estimand.** The average treatment effect

$$\text{ATE} = E[Y(1) - Y(0)],$$

the expected change in productivity if we moved everyone from untrained to trained. We will build the world so that the **true ATE is exactly 3.0**, then see which analyses recover it.

## 2 · Encode assumptions — a DAG as a data-generating process

Our assumed causal graph:

```
  experience (X1) ─┐        ┌─> Y
                   ├─> D ───┤
  motivation (X2) ─┘        └─> Y
          noise (X3) ─────────> Y   (affects Y only, not D)
```

`X1` and `X2` are **confounders** (they cause both `D` and `Y`). `X3` affects only `Y`. There is **no unmeasured confounding** in the true world — we will *pretend* otherwise later, in the sensitivity step, to stress-test the conclusion.

In [ ]:
N = 8000
TRUE_ATE = 3.0          # <-- the ground truth we must recover

# Confounders
X1 = RNG.normal(size=N)                  # experience
X2 = RNG.normal(size=N)                  # motivation
X3 = RNG.normal(size=N)                  # outcome-only driver

# Treatment: experienced & motivated people enroll more (propensity)
logit = -0.4 + 0.9 * X1 + 0.7 * X2
propensity = 1.0 / (1.0 + np.exp(-logit))
D = RNG.binomial(1, propensity)

# Outcome: TRUE effect of D is exactly TRUE_ATE; confounders push Y too
Y = (TRUE_ATE * D + 1.5 * X1 + 1.2 * X2 + 0.8 * X3
     + RNG.normal(size=N))

df = pd.DataFrame({'Y': Y, 'D': D, 'X1': X1, 'X2': X2, 'X3': X3,
                   'propensity': propensity})
print(f'n = {N},  treated = {D.mean():.1%},  TRUE ATE = {TRUE_ATE}')
print(f'overlap: propensity in [{propensity.min():.2f}, '
      f'{propensity.max():.2f}]  (positivity looks OK)')

### The naive comparison is biased

Before doing anything clever, compare trained vs. untrained directly. Because enrollees were already more experienced and motivated, the raw gap **overstates** the effect — this is the confounding we must remove.

In [ ]:
naive = df.loc[df.D == 1, 'Y'].mean() - df.loc[df.D == 0, 'Y'].mean()
print(f'Naive difference in means : {naive:.3f}')
print(f'True ATE                  : {TRUE_ATE:.3f}')
print(f'Bias from confounding     : {naive - TRUE_ATE:+.3f}  '
      '(inflated, as expected)')
assert naive > TRUE_ATE + 0.3, 'naive estimate should be biased upward'

## 3 · Identify — the back-door adjustment set

To read the `D → Y` effect we must block every back-door path. The back-door paths run `D ← X1 → Y` and `D ← X2 → Y`, so the adjustment set is **{X1, X2}**.

- `X1`, `X2`: **confounders → adjust.**
- `X3`: affects only `Y`, opens no back-door path. Adjusting is harmless (and slightly improves precision), but it is *not required* for identification.

With {X1, X2} measured and overlap holding, the ATE is **identified** by back-door adjustment.

## 4 · Estimate with TWO methods

We estimate the same identified ATE two independent ways and check they **agree with each other** and **recover the truth**:

- **(A) OLS regression adjustment** — include the back-door set.
- **(B) IPW** — inverse-probability weighting with a logistic propensity model.

Two methods that disagree are a red flag; two that agree on the truth are reassuring.

In [ ]:
# --- (A) OLS regression adjustment on the back-door set {X1, X2} ---
Xmat = sm.add_constant(df[['D', 'X1', 'X2']].to_numpy())
ols = sm.OLS(df['Y'].to_numpy(), Xmat).fit()
ate_ols = ols.params[1]                 # coefficient on D
se_ols  = ols.bse[1]
print(f'(A) OLS adjustment ATE = {ate_ols:.3f}  '
      f'(95% CI {ate_ols - 1.96*se_ols:.2f} to {ate_ols + 1.96*se_ols:.2f})')
assert abs(ate_ols - TRUE_ATE) < 0.15, 'OLS should recover ~3.0'

In [ ]:
# --- (B) IPW with a logistic propensity model on {X1, X2} ---
from sklearn.linear_model import LogisticRegression

Xc = df[['X1', 'X2']].to_numpy()
Dv = df['D'].to_numpy()
Yv = df['Y'].to_numpy()

ps = LogisticRegression().fit(Xc, Dv).predict_proba(Xc)[:, 1]
ps = np.clip(ps, 0.02, 0.98)            # trim to respect positivity
w  = np.where(Dv == 1, 1.0 / ps, 1.0 / (1.0 - ps))

ate_ipw = (np.average(Yv[Dv == 1], weights=w[Dv == 1])
           - np.average(Yv[Dv == 0], weights=w[Dv == 0]))
print(f'(B) IPW ATE        = {ate_ipw:.3f}')
print(f'    OLS ATE        = {ate_ols:.3f}')
print(f'    TRUE ATE       = {TRUE_ATE:.3f}')
assert abs(ate_ipw - TRUE_ATE) < 0.20, 'IPW should recover ~3.0'
assert abs(ate_ipw - ate_ols) < 0.25, 'the two methods should agree'

### 🔧 Exercise 4.1 — add a third estimator (cross-fitted DML)

Two methods agree; a third makes the result harder to dismiss. Implement **double / debiased machine learning** by the partialling-out trick, done with *linear* nuisance models and **5-fold cross-fitting**:

1. Cross-fit predictions of `Y` from `{X1, X2}` and of `D` from `{X1, X2}`.
2. Form residuals `Ỹ = Y − Ŷ` and `D̃ = D − D̂`.
3. The DML ATE is the slope of `Ỹ` on `D̃`: `sum(D̃·Ỹ) / sum(D̃·D̃)`.

Fill in the `# TODO`s. The skeleton runs as-is (it just falls back to the OLS estimate) so the notebook never breaks.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold

Xc = df[['X1', 'X2']].to_numpy()
Dv = df['D'].to_numpy().astype(float)
Yv = df['Y'].to_numpy()

kf = KFold(n_splits=5, shuffle=True, random_state=0)
Y_res = np.zeros(N)
D_res = np.zeros(N)
done = False   # flip to True once you fill the loop in

for train_idx, test_idx in kf.split(Xc):
    # TODO: fit LinearRegression of Y on Xc using train_idx,
    #       then store the residual on test_idx into Y_res[test_idx]
    # TODO: do the same for D into D_res[test_idx]
    # TODO: set done = True
    pass

# Fallback so the skeleton runs even before you fill it in:
if done:
    ate_dml = np.sum(D_res * Y_res) / np.sum(D_res * D_res)
else:
    ate_dml = ate_ols   # placeholder until you implement the loop
print(f'(skeleton) DML ATE = {ate_dml:.3f}')

### ✅ Solution 4.1

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=0)
Y_res = np.zeros(N)
D_res = np.zeros(N)

for train_idx, test_idx in kf.split(Xc):
    m_y = LinearRegression().fit(Xc[train_idx], Yv[train_idx])
    m_d = LinearRegression().fit(Xc[train_idx], Dv[train_idx])
    Y_res[test_idx] = Yv[test_idx] - m_y.predict(Xc[test_idx])
    D_res[test_idx] = Dv[test_idx] - m_d.predict(Xc[test_idx])

ate_dml = np.sum(D_res * Y_res) / np.sum(D_res * D_res)
print(f'(A) OLS  ATE = {ate_ols:.3f}')
print(f'(B) IPW  ATE = {ate_ipw:.3f}')
print(f'(C) DML  ATE = {ate_dml:.3f}')
print(f'    TRUE ATE = {TRUE_ATE:.3f}')
assert abs(ate_dml - TRUE_ATE) < 0.15, 'DML should recover ~3.0'
print('\nThree independent estimators, one truth. That is a result '
      'you can defend.')

## 5 · Validate — robustness / multiverse loop

Instead of defending one specification, run the **grid** of reasonable ones and look at the *distribution* of estimates. We vary:

- whether we adjust for `X2` (a real confounder),
- whether we adjust for `X3` (an outcome-only variable, optional),
- whether we add a quadratic term in `X1` (functional form).

`X1` is always included. The key lesson: specifications that include the **X2 confounder** cluster on the truth; dropping it drifts away.

In [ ]:
from itertools import product

rows = []
for use_x2, use_x3, quad in product([0, 1], [0, 1], [0, 1]):
    cols = ['D', 'X1'] + (['X2'] if use_x2 else []) + \
           (['X3'] if use_x3 else [])
    Xm = df[cols].to_numpy().astype(float)
    if quad:
        Xm = np.column_stack([Xm, df['X1'].to_numpy() ** 2])
    est = sm.OLS(df['Y'].to_numpy(), sm.add_constant(Xm)).fit().params[1]
    rows.append({'adj_X2': use_x2, 'adj_X3': use_x3,
                 'quad_X1': quad, 'ate': est})

multiverse = pd.DataFrame(rows).sort_values('ate').reset_index(drop=True)
print(multiverse)
print(f'\nFull multiverse range: '
      f'{multiverse.ate.min():.2f} to {multiverse.ate.max():.2f}')

In [ ]:
# Split the multiverse by whether the X2 confounder was adjusted for
with_x2 = multiverse.loc[multiverse.adj_X2 == 1, 'ate']
without_x2 = multiverse.loc[multiverse.adj_X2 == 0, 'ate']
print(f'Specs adjusting for X2   : mean {with_x2.mean():.3f}  '
      f'(range {with_x2.min():.2f}-{with_x2.max():.2f})')
print(f'Specs OMITTING X2        : mean {without_x2.mean():.3f}  '
      f'(range {without_x2.min():.2f}-{without_x2.max():.2f})')
print(f'True ATE                 : {TRUE_ATE:.3f}')

# The correctly-specified family recovers the truth; the rest is bias.
assert abs(with_x2.mean() - TRUE_ATE) < 0.15
assert without_x2.mean() > with_x2.mean() + 0.3
print('\n-> Robust WITHIN the correctly-identified specifications; '
      'the spread is driven by dropping a real confounder.')

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()
colors = ['#2A9D8F' if a else '#C0504D' for a in multiverse.adj_X2]
ax.scatter(multiverse.ate, range(len(multiverse)), c=colors, s=70,
           zorder=3)
ax.axvline(TRUE_ATE, color='#1F2A44', ls='--', label='true ATE = 3.0')
ax.set_xlabel('estimated ATE')
ax.set_ylabel('specification (sorted)')
ax.set_title('Multiverse: teal = adjusts for X2 confounder, '
             'red = omits it')
ax.legend(loc='lower right')
# figure created (no blocking show); it would be embedded in the report

### Sensitivity — how strong would unmeasured confounding have to be?

Now *pretend* `X2` was never measured, so our only adjusted estimate omits it and is biased. The skeptic says 'unmeasured confounding!' We answer **quantitatively** with an **E-value**: the minimum strength (on the risk-ratio scale) that a hidden confounder would need with *both* treatment and outcome to fully explain the observed effect away. A large E-value means the skeptic needs an implausibly strong hidden cause.

In [ ]:
def e_value(rr):
    """E-value for an observed risk ratio rr (VanderWeele & Ding 2017)."""
    rr = max(rr, 1.0 / rr)            # work on the >= 1 side
    return rr + np.sqrt(rr * (rr - 1.0))

# Translate a standardized mean effect into an approximate risk ratio
# (a common rough conversion: RR ~ exp(0.91 * standardized effect)).
sd_y = df['Y'].std()
rr_observed = float(np.exp(0.91 * (ate_ols / sd_y)))
ev = e_value(rr_observed)
print(f'Approx. observed risk ratio : {rr_observed:.3f}')
print(f'E-value                     : {ev:.3f}')
print(f'Interpretation: an unmeasured confounder would need RR >= '
      f'{ev:.2f}')
print('with BOTH treatment and outcome to explain the effect away.')
assert ev > 1.0, 'a non-null effect should have an E-value above 1'

### 🔧 Exercise 5.1 — quantify the bias from dropping a confounder

Estimate the ATE while **omitting `X2`** (simulating an unmeasured confounder), and compute how much bias that introduces relative to the truth. Then state, in a comment, whether the bias is large enough that you would worry. Fill in the `# TODO`s; the skeleton runs.

In [ ]:
# TODO: regress Y on [D, X1] only (X2 omitted), pull the D coefficient
Xm_omit = sm.add_constant(df[['D', 'X1']].to_numpy())
ate_omit = ...        # TODO: fit OLS and take .params[1]
# bias_omit = ...     # TODO: ate_omit - TRUE_ATE
# print(ate_omit, bias_omit)

# Fallback so the skeleton runs before you fill it in:
if ate_omit is ...:
    ate_omit = ate_ols
print(f'(skeleton) ATE omitting X2 = {ate_omit:.3f}')

### ✅ Solution 5.1

In [ ]:
Xm_omit = sm.add_constant(df[['D', 'X1']].to_numpy())
ate_omit = sm.OLS(df['Y'].to_numpy(), Xm_omit).fit().params[1]
bias_omit = ate_omit - TRUE_ATE
print(f'ATE adjusting for {{X1, X2}} : {ate_ols:.3f}  (correct)')
print(f'ATE omitting X2            : {ate_omit:.3f}  (confounded)')
print(f'Bias from the hidden X2    : {bias_omit:+.3f}')
assert ate_omit > ate_ols + 0.3, 'omitting a real confounder inflates the ATE'
print('\nThe omitted confounder inflates the effect — exactly the '
      'scenario the E-value is built to reason about.')

## 6 · Communicate the result

The analysis is only useful if a decision-maker can act on it. Here is the **three-sentence stakeholder summary** — effect, uncertainty, caveat — with no jargon. (In your capstone, paste the real numbers from the cells above.)

In [ ]:
lo = ate_ols - 1.96 * se_ols
hi = ate_ols + 1.96 * se_ols
summary = (
    f'1. On average, the training program raised productivity by about '
    f'{ate_ols:.1f} points per employee.\n'
    f'2. Our best estimate could reasonably lie between '
    f'{lo:.1f} and {hi:.1f}, so the program clearly helps, though the '
    f'exact size is uncertain.\n'
    f'3. This holds if enrollees and non-enrollees were comparable '
    f'once we account for experience and motivation; if some other '
    f'unmeasured trait drove both enrolling and productivity '
    f'(E-value ~ {ev:.1f}), the true effect would be smaller.'
)
print(summary)

> **Effect, uncertainty, caveat — three sentences, no p-values.**
>
> Notice what the summary does *not* do: it does not say 'statistically significant', it does not hide the assumption, and it does not pretend the point estimate is exact. That honesty is the whole point of the capstone.

## Wrap-up & self-check

You just ran a complete, defensible causal analysis end to end:

- **Estimand** stated (ATE of training on productivity).
- **Assumptions** encoded as a DAG-shaped DGP with a known truth (3.0).
- **Identified** via the back-door set {X1, X2}.
- **Estimated** three ways — OLS, IPW, DML — all recovering ~3.0 and agreeing.
- **Validated** with a multiverse loop (robust within the correctly identified family) and an **E-value** sensitivity check.
- **Communicated** the effect, its uncertainty, and the load-bearing caveat in three plain sentences.
- **Reproducible**: one fixed seed (`RNG`), a printed environment log.

**This is the skeleton of your capstone.** Replace the simulated world with your real question and data, keep every step, and you have a submission you can defend in front of a reviewer — and a decision-maker.

*There is no Week 16: from here, the syllabus is the world. Go do causal inference in practice.*